<a href="https://colab.research.google.com/github/Prktriesitout/Flight_screening_Agent/blob/main/Flight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

!pip install -q --upgrade openai requests


In [4]:

import requests
from getpass import getpass
from datetime import date, datetime
from zoneinfo import ZoneInfo
from typing import Optional, Literal
from urllib.parse import urlencode

from openai import OpenAI
from openai.lib.beta.agents import function_tool


# ==========================================
# 1. API CONFIGURATION
# ==========================================

import os
from google.colab import userdata

# Retrieve the keys from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI')
os.environ["SERPER_API_KEY"] = userdata.get('serper')


AGENT_ID = "agent_6afda51a7a588191b4b4a73b049dec77"

SERPER_COUNTRY = "in"

client = OpenAI(api_key = userdata.get('OpenAI'))

print("API configuration completed.")


# ==========================================
# 2. FLIGHT SEARCH FUNCTION
# ==========================================

@function_tool(
    name="find_cheapest_flights",
    description=(
        "Search Google-indexed flight information using the Serper API. "
        "Returns flight-related search results and a Google Flights link. "
        "Use after collecting all required travel details."
    )
)
def find_cheapest_flights(
    origin: str,
    destination: str,
    departure_date: str,
    return_date: Optional[str],
    adults: int,
    children: int,
    currency: str,
    cabin_class: Literal[
        "economy",
        "premium_economy",
        "business",
        "first"
    ]
) -> dict:
    """
    Search Google for flight information between two locations.
    """

    # Validate dates using Indian local time.
    today = datetime.now(
        ZoneInfo("Asia/Kolkata")
    ).date()

    try:
        departure = date.fromisoformat(departure_date)
    except ValueError:
        raise ValueError("Departure date must be YYYY-MM-DD.")

    if departure < today:
        raise ValueError("Departure date cannot be in the past.")

    if return_date is not None:

        try:
            returning = date.fromisoformat(return_date)
        except ValueError:
            raise ValueError("Return date must be YYYY-MM-DD.")

        if returning < departure:
            raise ValueError(
                "Return date cannot be before departure date."
            )

    # Validate passengers.
    if adults < 1:
        raise ValueError("At least one adult is required.")

    if children < 0:
        raise ValueError("Children cannot be negative.")

    # Validate currency.
    currency = currency.upper().strip()

    if len(currency) != 3:
        raise ValueError("Enter a valid 3-letter currency code.")

    # ======================================
    # Construct Google Search Query
    # ======================================

    query = (
        f"Google Flights flights from {origin} to {destination} "
        f"on {departure_date} "
        f"{adults} adults and {children} children "
        f"{cabin_class} class fares in {currency}"
    )

    if return_date:
        query += f" returning on {return_date}"
    else:
        query += " one way"

    # ======================================
    # Construct Google Flights URL
    # ======================================

    flight_query = (
        f"Flights from {origin} to {destination} "
        f"on {departure_date}"
    )

    if return_date:
        flight_query += f" returning on {return_date}"

    google_flights_url = (
        "https://www.google.com/travel/flights?"
        + urlencode({
            "q": flight_query,
            "curr": currency,
            "hl": "en"
        })
    )

    # ======================================
    # Call Serper API
    # ======================================

    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "q": query,
        "gl": SERPER_COUNTRY,
        "hl": "en",
        "num": 10
    }

    try:
        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

    except requests.RequestException as error:
        raise RuntimeError(
            f"Serper API request failed: {error}"
        )

    # ======================================
    # Extract Search Results
    # ======================================

    organic_results = data.get("organic", [])

    flights = []

    for item in organic_results:

        flights.append({
            "title": item.get("title"),
            "url": item.get("link"),
            "snippet": item.get("snippet"),
            "date": item.get("date"),
            "position": item.get("position")
        })

    # ======================================
    # Return to OpenAI Agent
    # ======================================

    return {
        "route": f"{origin} to {destination}",

        "departure_date": departure_date,

        "return_date": return_date,

        "passengers": {
            "adults": adults,
            "children": children
        },

        "cabin_class": cabin_class,

        "currency": currency,

        "search_results": flights,

        "google_flights_url": google_flights_url,

        "search_query": query,

        "source": "Serper Google Search API",

        "important_note": (
            "These are Google Search results, not a structured "
            "Google Flights inventory. Do not assume that every "
            "result is a currently bookable flight. Only report "
            "prices and airline details explicitly supported "
            "by the returned search results."
        )
    }


print("Flight search function registered.")


API configuration completed.
Flight search function registered.


In [5]:

# ==========================================
# 3. AGENT INSTRUCTIONS
# ==========================================

INSTRUCTIONS = """
You are Kabira, an intelligent and conversational
flight search assistant.

Your goal is to help users discover affordable flights.

CONVERSATION:

Collect the following information:

1. Origin
2. Destination
3. Departure date
4. Trip type: one-way or round-trip
5. Return date, if round-trip
6. Number of adults
7. Number of children
8. Preferred currency
9. Cabin class

DEFAULT VALUES:

- Adults: 1
- Children: 0
- Currency: INR
- Cabin class: economy

Ask the user for missing information.
Do not ask for information already provided.

Ask concise questions and avoid overwhelming the user.

DATE RULES:

- Use YYYY-MM-DD format when calling the function.
- Interpret dates using the current date.
- Never assume an ambiguous date.
- A one-way journey must have return_date = null.

FLIGHT SEARCH:

Once all required details are collected:

1. Call find_cheapest_flights.
2. Examine the returned search results.
3. Identify relevant flight-related information.
4. Provide source links.
5. Always provide the Google Flights search URL.

IMPORTANT ACCURACY RULES:

- Never invent airlines.
- Never invent departure or arrival times.
- Never invent ticket prices.
- Never invent flight availability.
- Never claim that a flight is the globally cheapest.
- Only mention prices explicitly supported by search results.
- Distinguish search snippets from confirmed fares.

Serper returns Google Search results, not a structured
Google Flights inventory.

If reliable fare information is unavailable, tell the
user that they should verify current prices on Google Flights.

RESPONSE STYLE:

Be friendly, concise and conversational.

Present flight information in an organised format.

Help users compare available information based on price,
route and travel requirements.
"""


# ==========================================
# 4. UPDATE EXISTING AGENT
# ==========================================

agent = client.beta.agents.update(
    AGENT_ID,

    name="Kabira",

    tools=[
        find_cheapest_flights.definition
    ],

    multi_agent={
        "enabled": False
    },

    instructions=INSTRUCTIONS,

    reasoning={
        "effort": "medium",
        "summary": "auto"
    },

    text={
        "format": {
            "type": "text"
        },
        "verbosity": "medium"
    }
)

print("Kabira updated successfully!")
print("Agent ID:", agent.id)
print("Model:", agent.model)


Kabira updated successfully!
Agent ID: agent_6afda51a7a588191b4b4a73b049dec77
Model: gpt-6.2-astra


In [6]:

# ==========================================
# 5. TOOL HANDLER REGISTRATION
# ==========================================

tool_handlers = {
    "find_cheapest_flights": find_cheapest_flights
}


# ==========================================
# 6. START AGENT SESSION
# ==========================================

def start_kabira():

    print("\n" + "=" * 50)
    print("       KABIRA - FLIGHT SEARCH ASSISTANT")
    print("=" * 50)

    initial_message = (
        "Hello Kabira! I want to find affordable flights. "
        "Please help me with the search."
    )

    with client.beta.agents.sessions.create(

        agent_id=AGENT_ID,

        environment={
            "type": "none"
        },

        input=initial_message,

        stream=True,

        tool_handlers=tool_handlers

    ).with_result_collection() as stream:

        result = stream.get_final_result()

    session_id = result.session_id

    print("\nKabira:")
    print(result.output_text)

    print("\nSession ID:", session_id)

    return session_id


# ==========================================
# 7. CONTINUE CONVERSATION
# ==========================================

def chat_with_kabira(session_id):

    while True:

        try:

            user_message = input("\nYou: ").strip()

            if user_message.lower() in [
                "exit",
                "quit",
                "stop"
            ]:
                print("\nKabira: Happy travelling! Goodbye.")
                break

            if not user_message:
                continue

            with client.beta.agents.sessions.stream(

                session_id,

                input=user_message,

                tool_handlers=tool_handlers

            ).with_result_collection() as stream:

                result = stream.get_final_result()

            print("\nKabira:")
            print(result.output_text)

        except KeyboardInterrupt:

            print("\nConversation stopped.")
            break

        except Exception as error:

            print("\nAn error occurred:")
            print(error)


# ==========================================
# 8. LAUNCH
# ==========================================

session_id = start_kabira()

chat_with_kabira(session_id)



       KABIRA - FLIGHT SEARCH ASSISTANT

Kabira:
Hi! Where are you flying from and to, and when would you like to leave? If it’s a round trip, include your return date. I’ll search for one adult in economy and show prices in INR unless you prefer otherwise.

Session ID: sess_05defbe481d81bae006afda8d48ee4819b9c20b884b034d3c7

You: Delhi

Kabira:
I’ll take Delhi as your departure city. Where would you like to fly, and on what date? Let me know if it’s one-way or include your return date for a round trip.

You: bankok

Kabira:
Delhi to Bangkok. When would you like to leave? Let me know if it’s one-way, or share your return date if it’s a round trip.

You: it is a round trip , i would like to book on 21 december and return tri[ on 28th dec

Kabira:
For **Delhi to Bangkok, 21–28 December 2026**, the fare search didn’t return prices. You can check current round-trip fares in INR for **one adult in economy** using the link below. Select **Cheapest** to compare options across both Bangkok ai